In [4]:
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

# =====================================================================
# IMPORTANT: Define your paths and parameters here before running
# =====================================================================
IMAGE_SIZE = (100, 100)
BATCH_SIZE = 32
train_dir = 'Training'
test_dir = 'Test'

# 1. Load the training dataset
print("Loading training dataset...")
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

# Load the test dataset 
print("Loading test dataset...")
test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

# 2. Extract the true number of classes found in the directory
NUM_CLASSES = len(train_ds.class_names) 
print(f"\nDetected {NUM_CLASSES} distinct classes in the training folder.")

# Performance optimization: Cache and prefetch datasets
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)

# 3. Define the CNN Architecture
model = models.Sequential([
    # Rescaling layer normalizes pixel values from [0, 255] to [0, 1]
    layers.Rescaling(1./255, input_shape=(100, 100, 3)),
    
    # First Convolutional block
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    # Second Convolutional block
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    # Third Convolutional block
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    
    # Flattening and Dense Layers
    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),  # Reduces overfitting
    
    # Output layer dynamically using the detected class count
    layers.Dense(NUM_CLASSES, activation='softmax')
])

# 4. Compile the Model
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# 5. Train the Model
EPOCHS = 10
history = model.fit(
    train_ds,          
    validation_data=test_ds,  
    epochs=EPOCHS
)

# 6. Evaluate Performance
print("\nEvaluating model on the test data...")
test_loss, test_acc = model.evaluate(test_ds) 
print(f"\nFinal Test Accuracy: {test_acc * 100:.2f}%")


Loading training dataset...
Found 35426 files belonging to 3 classes.
Loading test dataset...
Found 13810 files belonging to 3 classes.

Detected 3 distinct classes in the training folder.


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ rescaling_1 (Rescaling)              │ (None, 100, 100, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_3 (Conv2D)                    │ (None, 98, 98, 32)          │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_3 (MaxPooling2D)       │ (None, 49, 49, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_4 (Conv2D)                    │ (None, 47, 47, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_4 (MaxPooling2D)       │ (None, 23, 23, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_5 (Conv2D)                    │ (None, 21, 21, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_5 (MaxPooling2D)       │ (None, 10, 10, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_1 (Flatten)                  │ (None, 12800)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 256)                 │       3,277,056 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 3)                   │             771 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 3,371,075 (12.86 MB)

 Trainable params: 3,371,075 (12.86 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/10
1108/1108 ━━━━━━━━━━━━━━━━━━━━ 255s 185ms/step - accuracy: 0.9623 - loss: 0.0975 - val_accuracy: 0.8500 - val_loss: 1.1695
Epoch 2/10
1108/1108 ━━━━━━━━━━━━━━━━━━━━ 164s 148ms/step - accuracy: 0.9951 - loss: 0.0172 - val_accuracy: 0.8853 - val_loss: 1.1955
Epoch 3/10
1108/1108 ━━━━━━━━━━━━━━━━━━━━ 190s 171ms/step - accuracy: 0.9966 - loss: 0.0103 - val_accuracy: 0.9014 - val_loss: 1.6843
Epoch 4/10
1108/1108 ━━━━━━━━━━━━━━━━━━━━ 203s 172ms/step - accuracy: 0.9999 - loss: 2.1710e-04 - val_accuracy: 0.8600 - val_loss: 2.2152
Epoch 5/10
1108/1108 ━━━━━━━━━━━━━━━━━━━━ 172s 155ms/step - accuracy: 0.9962 - loss: 0.0131 - val_accuracy: 0.8673 - val_loss: 2.1217
Epoch 6/10
1108/1108 ━━━━━━━━━━━━━━━━━━━━ 171s 154ms/step - accuracy: 0.9984 - loss: 0.0063 - val_accuracy: 0.8770 - val_loss: 2.7289
Epoch 7/10
1108/1108 ━━━━━━━━━━━━━━━━━━━━ 167s 150ms/step - accuracy: 0.9975 - loss: 0.0107 - val_accuracy: 0.8828 - val_loss: 1.7593
Epoch 8/10
1108/1108 ━━━━━━━━━━━━━━━━━━━━ 171s 155ms/step 